# Roommate Compatibility Scorer — Jupyter Notebook + MySQL
### Student routine edition

This notebook compares students using their **daily routines and room-sharing habits**. It stores profiles in MySQL and calculates a simple, explainable compatibility score.

**Routine details included**
- Wake-up time and bedtime
- Sleep hours and alarm preference
- College/class start and end times
- Daily self-study hours and preferred study time
- Exam preparation habits
- Noise tolerance and lights-off time
- Nap, workout, meal schedule, and weekend routine
- Cleanliness, visitors, and smoking preference

> The score is a decision-support tool, not a guarantee that two students will get along. Students should review the reasons and discuss important preferences before choosing a roommate.

## 1. Install the required packages

Run this cell once. If Jupyter asks for a restart after installation, restart the kernel and continue.

In [ ]:
%pip install pandas matplotlib mysql-connector-python

## 2. Connect to MySQL

Make sure MySQL Server is running. Enter your MySQL password when prompted; it will not be shown on screen.

The notebook creates the `roommate_scorer` database and a `student_routines` table if they do not already exist.

In [ ]:
import getpass
import mysql.connector
from mysql.connector import Error
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

DB_HOST = "localhost"
DB_PORT = 3306
DB_USER = input("MySQL username (usually root): ").strip() or "root"
DB_PASSWORD = getpass.getpass("MySQL password: ")

server_connection = mysql.connector.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD
)
server_cursor = server_connection.cursor()
server_cursor.execute("CREATE DATABASE IF NOT EXISTS roommate_scorer")
server_cursor.close()
server_connection.close()

connection = mysql.connector.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database="roommate_scorer"
)
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS student_routines (
    student_id INT AUTO_INCREMENT PRIMARY KEY,
    student_name VARCHAR(100) NOT NULL,
    age INT,
    course VARCHAR(100),
    wake_time TIME NOT NULL,
    bedtime TIME NOT NULL,
    sleep_hours DECIMAL(3,1) NOT NULL,
    alarm_preference VARCHAR(30) NOT NULL,
    college_start TIME,
    college_end TIME,
    self_study_hours DECIMAL(3,1) NOT NULL,
    preferred_study_time VARCHAR(30) NOT NULL,
    exam_study_style VARCHAR(40) NOT NULL,
    noise_tolerance VARCHAR(30) NOT NULL,
    lights_off_time TIME,
    nap_habit VARCHAR(30) NOT NULL,
    workout_habit VARCHAR(30) NOT NULL,
    meal_schedule VARCHAR(30) NOT NULL,
    weekend_routine VARCHAR(40) NOT NULL,
    cleanliness VARCHAR(30) NOT NULL,
    visitor_preference VARCHAR(30) NOT NULL,
    smoking_preference VARCHAR(30) NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")
connection.commit()
print("Connected to MySQL. Database and table are ready.")

## 3. Understand the compatibility weights

The score is out of 100. Sleep and daily routine receive the largest weights because mismatched sleep schedules can affect roommates every day. You can adjust the weights if your project needs a different priority.

In [ ]:
WEIGHTS = {
    "sleep_schedule": 20,
    "college_schedule": 10,
    "study_hours": 15,
    "study_time": 10,
    "noise_tolerance": 10,
    "lights_off": 5,
    "alarm": 5,
    "nap": 5,
    "workout": 3,
    "meal_schedule": 3,
    "weekend_routine": 5,
    "cleanliness": 5,
    "visitors": 3,
    "smoking": 1
}

assert sum(WEIGHTS.values()) == 100
pd.DataFrame({"Factor": WEIGHTS.keys(), "Weight (%)": WEIGHTS.values()})

## 4. Add student profiles

Edit the example dictionary below with a student's real preferences, then run the cell. Run it again with a different name and preferences to add more students.

Use 24-hour time in `HH:MM` format, such as `06:30` or `23:00`.

In [ ]:
def add_student(profile):
    required = [
        "student_name", "wake_time", "bedtime", "sleep_hours",
        "alarm_preference", "self_study_hours", "preferred_study_time",
        "exam_study_style", "noise_tolerance", "nap_habit",
        "workout_habit", "meal_schedule", "weekend_routine",
        "cleanliness", "visitor_preference", "smoking_preference"
    ]
    missing = [field for field in required if field not in profile]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")

    fields = [
        "student_name", "age", "course", "wake_time", "bedtime", "sleep_hours",
        "alarm_preference", "college_start", "college_end", "self_study_hours",
        "preferred_study_time", "exam_study_style", "noise_tolerance",
        "lights_off_time", "nap_habit", "workout_habit", "meal_schedule",
        "weekend_routine", "cleanliness", "visitor_preference", "smoking_preference"
    ]
    values = [profile.get(field) for field in fields]
    placeholders = ", ".join(["%s"] * len(fields))
    query = f"INSERT INTO student_routines ({', '.join(fields)}) VALUES ({placeholders})"
    cursor.execute(query, values)
    connection.commit()
    print(f"Saved profile for {profile['student_name']} (ID: {cursor.lastrowid})")

example_student = {
    "student_name": "Student A",
    "age": 20,
    "course": "BCA Data Science",
    "wake_time": "06:30",
    "bedtime": "23:00",
    "sleep_hours": 7.5,
    "alarm_preference": "One alarm",
    "college_start": "09:30",
    "college_end": "17:00",
    "self_study_hours": 2.0,
    "preferred_study_time": "Evening",
    "exam_study_style": "Daily preparation",
    "noise_tolerance": "Low",
    "lights_off_time": "23:00",
    "nap_habit": "Sometimes",
    "workout_habit": "Morning",
    "meal_schedule": "Regular",
    "weekend_routine": "Study and relax",
    "cleanliness": "High",
    "visitor_preference": "Occasional visitors",
    "smoking_preference": "Non-smoking room"
}

# Run this line only when you want to add the example profile.
# Change the name before running it again to avoid duplicate example entries.
# add_student(example_student)

## 5. View saved student profiles

This reads the profiles from MySQL. Time fields are shown in a table for easy comparison.

In [ ]:
def load_students():
    return pd.read_sql("SELECT * FROM student_routines ORDER BY student_id", connection)

students = load_students()
print(f"Profiles saved: {len(students)}")
display(students)

## 6. Compatibility scoring logic

For each factor, students receive:
- **100 points** when preferences match closely
- **50 points** for a moderate difference
- **0 points** for a strong mismatch

Sleep/wake times are compared as clock times. A smaller time difference means a better match. The final score is a weighted average.

In [ ]:
def time_minutes(value):
    if value is None or pd.isna(value):
        return None
    if hasattr(value, "hour"):
        return value.hour * 60 + value.minute
    value = str(value)
    parts = value.split(":")
    return int(parts[0]) * 60 + int(parts[1])

def circular_time_similarity(a, b, good_gap=30, bad_gap=180):
    ma, mb = time_minutes(a), time_minutes(b)
    if ma is None or mb is None:
        return None
    gap = abs(ma - mb)
    gap = min(gap, 1440 - gap)
    if gap <= good_gap:
        return 100
    if gap >= bad_gap:
        return 0
    return round(100 * (bad_gap - gap) / (bad_gap - good_gap), 1)

def numeric_similarity(a, b, good_gap, bad_gap):
    if pd.isna(a) or pd.isna(b):
        return None
    gap = abs(float(a) - float(b))
    if gap <= good_gap:
        return 100
    if gap >= bad_gap:
        return 0
    return round(100 * (bad_gap - gap) / (bad_gap - good_gap), 1)

def category_similarity(a, b, partial_pairs=None):
    if pd.isna(a) or pd.isna(b):
        return None
    if str(a).strip().lower() == str(b).strip().lower():
        return 100
    if partial_pairs:
        pair = {str(a).strip().lower(), str(b).strip().lower()}
        if pair in partial_pairs:
            return 50
    return 0

def score_pair(a, b):
    comparisons = {
        "sleep_schedule": circular_time_similarity(a["wake_time"], b["wake_time"], 30, 180) * 0.5
                          + circular_time_similarity(a["bedtime"], b["bedtime"], 30, 180) * 0.5,
        "college_schedule": None,
        "study_hours": numeric_similarity(a["self_study_hours"], b["self_study_hours"], 1, 4),
        "study_time": category_similarity(a["preferred_study_time"], b["preferred_study_time"]),
        "noise_tolerance": category_similarity(a["noise_tolerance"], b["noise_tolerance"]),
        "lights_off": circular_time_similarity(a["lights_off_time"], b["lights_off_time"], 30, 180),
        "alarm": category_similarity(a["alarm_preference"], b["alarm_preference"]),
        "nap": category_similarity(a["nap_habit"], b["nap_habit"]),
        "workout": category_similarity(a["workout_habit"], b["workout_habit"]),
        "meal_schedule": category_similarity(a["meal_schedule"], b["meal_schedule"]),
        "weekend_routine": category_similarity(a["weekend_routine"], b["weekend_routine"]),
        "cleanliness": category_similarity(a["cleanliness"], b["cleanliness"]),
        "visitors": category_similarity(a["visitor_preference"], b["visitor_preference"]),
        "smoking": category_similarity(a["smoking_preference"], b["smoking_preference"])
    }

    # Compare college start/end times when both students have entered them.
    start_score = circular_time_similarity(a["college_start"], b["college_start"], 30, 180)
    end_score = circular_time_similarity(a["college_end"], b["college_end"], 30, 180)
    if start_score is not None and end_score is not None:
        comparisons["college_schedule"] = (start_score + end_score) / 2

    total_weight = 0
    weighted_score = 0
    factor_scores = {}
    for factor, weight in WEIGHTS.items():
        score = comparisons.get(factor)
        if score is None or pd.isna(score):
            continue
        factor_scores[factor] = round(float(score), 1)
        weighted_score += float(score) * weight
        total_weight += weight

    final_score = round(weighted_score / total_weight, 1) if total_weight else 0
    return final_score, factor_scores

def compatibility_label(score):
    if score >= 85:
        return "Excellent match"
    if score >= 70:
        return "Good match"
    if score >= 50:
        return "Moderate match"
    return "Low match"

## 7. Compare all possible roommate pairs

This produces a ranked list. If you have fewer than two profiles, add another student first.

In [ ]:
def compare_all_pairs(df):
    results = []
    records = df.to_dict("records")
    for i in range(len(records)):
        for j in range(i + 1, len(records)):
            a, b = records[i], records[j]
            score, factor_scores = score_pair(a, b)
            results.append({
                "Student 1": a["student_name"],
                "Student 2": b["student_name"],
                "Compatibility (%)": score,
                "Recommendation": compatibility_label(score),
                "Strongest matching factors": ", ".join(
                    name.replace("_", " ") for name, val in factor_scores.items() if val >= 85
                ) or "No strong matches identified",
                "Potential differences": ", ".join(
                    name.replace("_", " ") for name, val in factor_scores.items() if val < 50
                ) or "No major differences identified"
            })
    return pd.DataFrame(results).sort_values("Compatibility (%)", ascending=False).reset_index(drop=True) if results else pd.DataFrame()

students = load_students()
matches = compare_all_pairs(students)
display(matches)

## 8. Visualize compatibility scores

The chart helps you quickly compare the highest-ranked pairs.

In [ ]:
if not matches.empty:
    chart_data = matches.head(15).copy()
    chart_data["Pair"] = chart_data["Student 1"] + " + " + chart_data["Student 2"]
    plt.figure(figsize=(10, max(4, len(chart_data) * 0.45)))
    plt.barh(chart_data["Pair"][::-1], chart_data["Compatibility (%)"][::-1])
    plt.xlabel("Compatibility (%)")
    plt.title("Top Roommate Compatibility Matches")
    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print("Add at least two student profiles to create a compatibility chart.")

## 9. Export results to CSV

This is useful for your project report or presentation.

In [ ]:
students = load_students()
matches = compare_all_pairs(students)
students.to_csv("student_routines_export.csv", index=False)
matches.to_csv("roommate_match_results.csv", index=False)
print("Exported student_routines_export.csv and roommate_match_results.csv")

## 10. Delete a profile (optional)

Check the ID before deleting. The delete action is permanent.

In [ ]:
# First display the IDs:
display(load_students()[["student_id", "student_name"]])

# Uncomment and replace 1 with the ID you want to delete:
# student_id_to_delete = 1
# cursor.execute("DELETE FROM student_routines WHERE student_id = %s", (student_id_to_delete,))
# connection.commit()
# print("Profile deleted.")

## 11. Close the database connection when finished

Run this cell at the end of your session.

In [ ]:
cursor.close()
connection.close()
print("MySQL connection closed.")